# DistilBERT fine-tuning: Product and Priority heads

Two separate single-task models (a joint multi-task head is a later stretch goal per the build plan). Sized for a free-tier Colab T4: sequence length capped at 256 tokens, mixed precision, gradient accumulation, checkpoints saved to Drive so a session disconnect doesn't lose progress.

**Before running**: Runtime -> Change runtime type -> T4 GPU. Then upload `model_ready_data.parquet` (from `scripts/export_training_data.py`, run locally) to your Google Drive and set `DRIVE_DIR` below to match where you put it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this to wherever you uploaded model_ready_data.parquet and where
# you want checkpoints/final models saved. Everything below reads/writes
# under this one directory.
DRIVE_DIR = '/content/drive/MyDrive/ticket-triage'

In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected -- go to Runtime > Change runtime type > T4 GPU before continuing.')

In [ ]:
import os

import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from transformers.trainer_utils import get_last_checkpoint
import matplotlib.pyplot as plt

MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 256

## Load the exported data

Same train/val/test split used for the TF-IDF baselines (`sql/006_train_val_test_split.sql`), exported via `scripts/export_training_data.py` -- narrative text, canonical product, priority bucket, split, and `complaint_id` (kept only as a row identifier for later error analysis, never used as a feature).

In [ ]:
df = pd.read_parquet(f'{DRIVE_DIR}/model_ready_data.parquet')
print(df.shape)
print(df['split'].value_counts())

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df = df[df['split'] == 'val'].reset_index(drop=True)

## Tokenizer

`padding='max_length'` (rather than dynamic padding) trades a little efficiency for simplicity and predictability -- every batch has the same shape, which matters more here than shaving off training time, given this notebook can't be debugged interactively the way the local scripts were.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(examples['narrative'], truncation=True, max_length=MAX_LENGTH, padding='max_length')

## Weighted Trainer

The Hugging Face `Trainer` has no built-in `class_weight='balanced'` the way scikit-learn does. This subclass overrides the loss computation to use a class-weighted `CrossEntropyLoss` instead -- same `'balanced'` weighting strategy used for the TF-IDF baselines, so the two are comparable.

In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'macro_f1': f1_score(labels, preds, average='macro'),
        'weighted_f1': f1_score(labels, preds, average='weighted'),
    }

## Training function

Shared by both heads -- only the label column and output directory differ. Checkpoints save to Drive every epoch (`save_strategy='epoch'`). Two layers of resumability, both aimed at picking up a paused or disconnected session without losing or redoing work:

- If `{DRIVE_DIR}/models/{head_name}/` already has a completed model (e.g. from an earlier session), it's loaded directly and training is skipped entirely -- rerunning the notebook later won't retrain a head that already finished.
- Otherwise, if `{DRIVE_DIR}/checkpoints/{head_name}/` has a saved epoch checkpoint (training was interrupted partway), `trainer.train(resume_from_checkpoint=...)` continues from there instead of restarting from the pretrained base model.

In [ ]:
def train_head(head_name, label_col, train_df, val_df):
    print(f'\n=== Training head: {head_name} ===')
    label_encoder = LabelEncoder()
    y_train = label_encoder.fit_transform(train_df[label_col])
    y_val = label_encoder.transform(val_df[label_col])
    n_classes = len(label_encoder.classes_)
    print(f'{n_classes} classes: {list(label_encoder.classes_)}')

    train_ds = Dataset.from_dict({'narrative': train_df['narrative'].tolist(), 'labels': y_train})
    val_ds = Dataset.from_dict({'narrative': val_df['narrative'].tolist(), 'labels': y_val})
    train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=['narrative'])
    val_ds = val_ds.map(tokenize_fn, batched=True, remove_columns=['narrative'])
    # Deliberately NOT calling .set_format('torch') here: datasets' own
    # torch formatter tries to import torchvision.io.VideoReader for an
    # unrelated video-data check whenever torchvision happens to be
    # loaded, and recent torchvision releases removed that class --
    # ImportError, even though nothing here touches video data. Trainer's
    # default data collator converts plain Python/numpy values to tensors
    # itself during batching, without going through that code path, so
    # leaving the datasets in their default (list-based) format sidesteps
    # the bug entirely rather than working around it.

    class_weights = compute_class_weight('balanced', classes=np.arange(n_classes), y=y_train)
    class_weights = torch.tensor(class_weights, dtype=torch.float)

    final_dir = f'{DRIVE_DIR}/models/{head_name}'
    checkpoint_dir = f'{DRIVE_DIR}/checkpoints/{head_name}'
    already_done = os.path.isdir(final_dir) and os.path.exists(f'{final_dir}/config.json')

    if already_done:
        print(f'Found a completed model at {final_dir} -- loading it instead of retraining.')
        model = AutoModelForSequenceClassification.from_pretrained(final_dir)
    else:
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=n_classes)

    args = TrainingArguments(
        output_dir=checkpoint_dir,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        learning_rate=2e-5,
        fp16=True,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        logging_steps=50,
        report_to='none',
    )

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
    )

    if already_done:
        print('Skipping training -- using the reloaded completed model as-is.')
    else:
        # Resume from the latest epoch checkpoint if one exists (e.g. the
        # previous session's runtime disconnected mid-training), rather
        # than silently starting over from the pretrained base model.
        last_checkpoint = get_last_checkpoint(checkpoint_dir) if os.path.isdir(checkpoint_dir) else None
        if last_checkpoint:
            print(f'Found an existing checkpoint, resuming from: {last_checkpoint}')
        else:
            print('No existing checkpoint found -- starting fresh.')
        trainer.train(resume_from_checkpoint=last_checkpoint)

        trainer.save_model(final_dir)
        tokenizer.save_pretrained(final_dir)
        print(f'Saved final model to {final_dir}')

    return trainer, label_encoder, val_ds, y_val

## Evaluation function

Same reporting shape as the baseline script (`scripts/train_baselines.py`) -- per-class precision/recall/F1, macro F1, weighted F1, confusion matrix -- so the two are directly comparable side by side.

In [ ]:
def evaluate_head(head_name, trainer, val_ds, y_val, label_encoder):
    preds = trainer.predict(val_ds)
    y_pred = np.argmax(preds.predictions, axis=-1)

    y_val_labels = label_encoder.inverse_transform(y_val)
    y_pred_labels = label_encoder.inverse_transform(y_pred)

    macro_f1 = f1_score(y_val_labels, y_pred_labels, average='macro')
    weighted_f1 = f1_score(y_val_labels, y_pred_labels, average='weighted')
    print(f'\n=== {head_name}: DistilBERT ===')
    print(f'macro F1: {macro_f1:.4f}   weighted F1: {weighted_f1:.4f}')
    print(classification_report(y_val_labels, y_pred_labels, zero_division=0))

    disp = ConfusionMatrixDisplay.from_predictions(
        y_val_labels, y_pred_labels, xticks_rotation='vertical', labels=label_encoder.classes_
    )
    disp.figure_.set_size_inches(8, 8)
    disp.figure_.tight_layout()
    fig_path = f'{DRIVE_DIR}/models/{head_name}_confusion_matrix.png'
    disp.figure_.savefig(fig_path, dpi=110)
    plt.show()
    print(f'Saved confusion matrix to {fig_path}')
    return macro_f1, weighted_f1

## Product head

In [ ]:
product_trainer, product_le, product_val_ds, product_y_val = train_head(
    'product', 'product', train_df, val_df
)

In [ ]:
product_macro_f1, product_weighted_f1 = evaluate_head(
    'product', product_trainer, product_val_ds, product_y_val, product_le
)

## Priority head

In [ ]:
priority_trainer, priority_le, priority_val_ds, priority_y_val = train_head(
    'priority', 'priority_bucket', train_df, val_df
)

In [ ]:
priority_macro_f1, priority_weighted_f1 = evaluate_head(
    'priority', priority_trainer, priority_val_ds, priority_y_val, priority_le
)

## Summary

Compare these macro F1 numbers directly against the baselines:

| Head | Model | macro F1 |
|---|---|---|
| Product | LogReg (baseline) | 0.575 |
| Product | XGBoost (baseline) | 0.571 |
| Product | **DistilBERT** | *(printed above)* |
| Priority | LogReg (baseline) | 0.405 |
| Priority | XGBoost (baseline) | 0.394 |
| Priority | **DistilBERT** | *(printed above)* |

The specific number to look at first for Priority: how many of the High-priority validation examples DistilBERT gets right, versus the 0/34 both baselines managed. Final models are saved to `{DRIVE_DIR}/models/product/` and `{DRIVE_DIR}/models/priority/` -- download these (or note the macro/weighted F1 and per-class numbers) to bring back into the main project for the evaluation report.